In [2]:
import pandas as pd
import os

In [8]:
df = pd.read_csv('../../data/raw_data/cardekho_listings_delhi.csv')

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7536 entries, 0 to 7535
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Title        7536 non-null   str  
 1   Link         7536 non-null   str  
 2   Price        7530 non-null   str  
 3   Information  7536 non-null   str  
 4   Image        7535 non-null   str  
 5   Year         7536 non-null   int64
 6   Brand        7536 non-null   str  
 7   Location     7536 non-null   str  
dtypes: int64(1), str(7)
memory usage: 471.1 KB


In [19]:
df.columns

Index(['Title', 'Link', 'Price', 'Information', 'Image', 'Year', 'Brand',
       'Location'],
      dtype='str')

In [14]:
df.head()

,Title,Link,Price,Information,Image,Year,Brand,Location
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,"₹ 231,000","60,000 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon"
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","58,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi"
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","41,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi"
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,"₹ 1,092,000","53,366 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad"
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,"₹ 830,000","1,46,000 kms • Diesel • Manual",https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida


In [37]:
units = set()

def handle_information(data):
    data = data.replace(' • ', '')
    parts = data.split()
    
    distance = parts[0].replace(',', '')
    unit = parts[1]
    units.add(unit)

    if len(parts) == 2:
        fuel_type,vehicle_type= None,None
    elif len(parts) == 3:
        fuel_type, vehicle_type = parts[2], None
    elif len(parts) == 4:
        fuel_type, vehicle_type = parts[2], parts[3]
    else:
        raise ValueError(f"Unexpected format: {parts}")
    
    return (int(distance), fuel_type, vehicle_type)

df[['Kms Covered', 'Fuel Type', 'Type']] = df['Information'].apply(handle_information).apply(pd.Series)

In [40]:
df.head()

,Title,Link,Price,Information,Image,Year,Brand,Location,Kms Covered,Fuel Type,Type
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,"₹ 231,000","60,000 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon",60000.0,Petrol,Automatic
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","58,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi",58000.0,Petrol,Manual
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,"₹ 575,000","41,000 kms • Petrol • Manual",https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi",41000.0,Petrol,Manual
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,"₹ 1,092,000","53,366 kms • Petrol • Automatic",https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad",53366.0,Petrol,Automatic
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,"₹ 830,000","1,46,000 kms • Diesel • Manual",https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida,146000.0,Diesel,Manual


In [41]:
units

{'kms'}

In [45]:
df['Fuel Type'].unique()

<StringArray>
['Petrol', 'Diesel', 'CNG', nan, 'Electric']
Length: 5, dtype: str

In [46]:
df['Type'].unique()

<StringArray>
['Automatic', 'Manual', nan]
Length: 3, dtype: str

In [54]:
df['Price'].isna().sum()

np.int64(6)

In [57]:
df =df.dropna(subset=['Price']).reset_index()

In [59]:
df.drop('index',axis = 1,inplace = True)

In [61]:
def refine_price(data):
    if type(data) == str:
        data = data.replace('₹','').replace(',','')
        return float(data)
    return data

df['Price'] = df['Price'].apply(refine_price)

In [63]:
df.drop('Information',inplace=True,axis = 1)

In [64]:
df.head()

,Title,Link,Price,Image,Year,Brand,Location,Kms Covered,Fuel Type,Type
0,2014 Maruti Suzuki Celerio VXI AT,https://www.cardekho.com/used-car-details/used...,231000.0,https://images10.gaadi.com/usedcar_image/49941...,2014,Maruti,"Badshahpur, Gurgaon",60000.0,Petrol,Automatic
1,2019 Maruti Suzuki Ciaz Zeta BSIV,https://www.cardekho.com/used-car-details/used...,575000.0,https://images10.gaadi.com/usedcar_image/51632...,2019,Maruti,"Lajpat Nagar, New Delhi",58000.0,Petrol,Manual
2,2018 Maruti Suzuki Ciaz Delta BSIV,https://www.cardekho.com/used-car-details/used...,575000.0,https://images10.gaadi.com/usedcar_image/50705...,2018,Maruti,"Lajpat Nagar, New Delhi",41000.0,Petrol,Manual
3,2022 Maruti Suzuki XL6 Alpha AT BSVI,https://www.cardekho.com/buy-used-car-details/...,1092000.0,https://images10.gaadi.com/usedcar_image/51622...,2022,Maruti,"Indirapuram, Ghaziabad",53366.0,Petrol,Automatic
4,2019 Maruti Suzuki Ertiga 1.5 VDI,https://www.cardekho.com/used-car-details/used...,830000.0,https://images10.gaadi.com/usedcar_image/50806...,2019,Maruti,Greater Noida,146000.0,Diesel,Manual


In [65]:
df.to_csv('../../data/cleaned_data/cardekho_cleaned_data.csv')